# Tahap 1 — Business & Data Understanding
### Proyek: Prediksi Customer Churn pada Perusahaan Telekomunikasi

Notebook ini adalah langkah pertama dari alur kerja data science untuk proyek
prediksi churn pelanggan Telco. Fokus tahap ini **bukan** membersihkan data
atau membuat model, melainkan:

1. Memahami konteks bisnis dan menuliskan ulang pertanyaan bisnis dengan
   kata-kata sendiri.
2. Menentukan target prediksi.
3. Memeriksa struktur data mentah: tipe kolom, statistik deskriptif, dan
   jumlah missing value — sebagai dasar untuk perencanaan cleaning di
   Tahap 2.

## 1. Konteks Bisnis (ditulis ulang dengan kata sendiri)

Perusahaan telko ini kehilangan pelanggan setiap bulan (*churn*). Biaya untuk
mendapatkan pelanggan baru jauh lebih mahal dibanding mempertahankan
pelanggan yang sudah ada, jadi kalau tim retensi tahu **lebih awal** siapa
saja pelanggan yang kemungkinan besar akan berhenti, mereka bisa proaktif
menawarkan promo/insentif sebelum pelanggan itu benar-benar pergi.

Jadi masalah bisnisnya bukan sekadar "prediksi angka", tapi: **siapa yang
harus dihubungi duluan oleh tim retensi, dan kenapa mereka berisiko pergi?**

## 2. Pertanyaan Bisnis (versi saya sendiri)

| # | Pertanyaan asli di brief | Versi saya |
|---|---|---|
| 1 | Faktor apa yang paling berhubungan dengan churn? | Karakteristik/perilaku pelanggan seperti apa yang paling sering muncul pada pelanggan yang berhenti — apakah soal jenis kontrak, lama berlangganan, biaya bulanan, atau jenis layanan yang dipakai? |
| 2 | Bisakah dibuat model prediksi probabilitas churn? | Bisakah kita membuat sistem skor risiko (0–1) per pelanggan, bukan cuma label Yes/No, supaya tim retensi bisa memprioritaskan pelanggan dengan skor risiko tertinggi? |
| 3 | Segmen mana yang paling berisiko & rekomendasi apa? | Kalau kita kelompokkan pelanggan berdasarkan kombinasi kontrak/tenure/layanan, kelompok mana yang churn rate-nya jauh di atas rata-rata, dan tindakan konkret apa yang bisa diambil tim retensi untuk tiap kelompok itu? |

## 3. Menentukan Target

- **Kolom target:** `Churn` (Yes / No).
- **Jenis masalah:** klasifikasi biner.
- **Kelas positif (yang ingin ditangkap):** `Yes` — pelanggan yang churn.
  Ini penting karena metrik yang dipakai nanti (recall, precision, F1 di
  Tahap 6) akan dihitung terhadap kelas `Yes`, bukan `No`.
- **Unit analisis:** 1 baris = 1 pelanggan (`customerID` sebagai ID, dibuang
  saat modeling karena tidak punya nilai prediktif).

## 4. Setup & Load Data

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 25)
pd.set_option("display.width", 120)

DATA_PATH = "../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv"
df = pd.read_csv(DATA_PATH)

print(f"Jumlah baris : {df.shape[0]}")
print(f"Jumlah kolom : {df.shape[1]}")

Jumlah baris : 7043
Jumlah kolom : 21


Sesuai brief: dataset seharusnya berisi **7.043 baris** dan **21 kolom** — dicek langsung di atas untuk memastikan file yang di-load memang benar.

### Sekilas isi data

In [2]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## 5. Struktur Data

Cek tipe data tiap kolom dan jumlah non-null value dengan `.info()`.

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

**Catatan awal dari `.info()`:**

- Semua kolom terbaca sebagai `object` (teks) kecuali `SeniorCitizen`
  (int64), `tenure` (int64), dan `MonthlyCharges` (float64).
- `TotalCharges` **seharusnya numerik** tapi terbaca sebagai `object` — ini
  konsisten dengan catatan di brief bahwa kolom ini tersimpan sebagai teks
  dan punya baris kosong. Akan diselidiki lebih lanjut di bawah, dan
  diperbaiki di **Tahap 2 (Data Cleaning)**.
- Tidak ada kolom yang menunjukkan non-null count lebih kecil dari jumlah
  baris total, artinya secara eksplisit (`NaN`) tidak ada missing value —
  tapi ini belum tentu berarti datanya benar-benar lengkap (lihat poin
  `TotalCharges` di atas).

## 6. Statistik Deskriptif

### Kolom numerik

In [4]:
df.describe()

,SeniorCitizen,tenure,MonthlyCharges
count,7043.000000,7043.000000,7043.000000
mean,0.162147,32.371149,64.761692
std,0.368612,24.559481,30.090047
min,0.000000,0.000000,18.250000
25%,0.000000,9.000000,35.500000
50%,0.000000,29.000000,70.350000
75%,0.000000,55.000000,89.850000
max,1.000000,72.000000,118.750000


Observasi cepat:

- `SeniorCitizen` walau numerik (0/1) sebenarnya kategori biner, bukan
  kuantitas — perlu ditangani secara berbeda saat feature engineering.
- `tenure` berkisar 0–72 bulan, dengan sebagian pelanggan tenure = 0
  (pelanggan baru) — ini yang nanti berkaitan dengan baris kosong di
  `TotalCharges`.
- `MonthlyCharges` berkisar sekitar 18–119 USD, distribusinya perlu dicek
  lebih lanjut saat EDA (Tahap 3).

### Kolom kategorikal

In [5]:
df.describe(include='object')

/var/folders/48/2cqh8zh14sqdjxcpdxg6pn440000gn/T/ipykernel_15253/87514550.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df.describe(include='object')


,customerID,gender,Partner,Dependents,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,TotalCharges,Churn
count,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043
unique,7043,2,2,2,2,3,3,3,3,3,3,3,3,3,2,4,6531,2
top,7590-VHVEG,Male,No,No,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,20.2,No
freq,1,3555,3641,4933,6361,3390,3096,3498,3088,3095,3473,2810,2785,3875,4171,2365,11,5174


Observasi cepat:

- `customerID` punya jumlah `unique` yang perlu disamakan dengan jumlah
  baris — kalau lebih kecil, berarti ada `customerID` duplikat (dicek di
  bagian 8).
- Kolom-kolom layanan (`OnlineSecurity`, `OnlineBackup`, `DeviceProtection`,
  `TechSupport`, `StreamingTV`, `StreamingMovies`) kemungkinan punya nilai
  ketiga selain Yes/No, misalnya `"No internet service"` — perlu dicek
  kategori uniknya.

In [6]:
service_cols = [
    "OnlineSecurity", "OnlineBackup", "DeviceProtection",
    "TechSupport", "StreamingTV", "StreamingMovies", "MultipleLines",
]
for col in service_cols:
    print(f"{col}: {sorted(df[col].unique())}")

OnlineSecurity: ['No', 'No internet service', 'Yes']
OnlineBackup: ['No', 'No internet service', 'Yes']
DeviceProtection: ['No', 'No internet service', 'Yes']
TechSupport: ['No', 'No internet service', 'Yes']
StreamingTV: ['No', 'No internet service', 'Yes']
StreamingMovies: ['No', 'No internet service', 'Yes']
MultipleLines: ['No', 'No phone service', 'Yes']


## 7. Missing Value per Kolom

In [7]:
missing = pd.DataFrame({
    "n_missing": df.isnull().sum(),
    "pct_missing": (df.isnull().sum() / len(df) * 100).round(2),
})
missing.sort_values("n_missing", ascending=False)

,n_missing,pct_missing
customerID,0,0.0
DeviceProtection,0,0.0
TotalCharges,0,0.0
MonthlyCharges,0,0.0
PaymentMethod,0,0.0
PaperlessBilling,0,0.0
Contract,0,0.0
StreamingMovies,0,0.0
StreamingTV,0,0.0
TechSupport,0,0.0


Secara eksplisit **tidak ada `NaN`** di kolom manapun. Tapi seperti
disinggung di brief, `TotalCharges` disimpan sebagai teks dan punya
**baris kosong berupa string spasi (`" "`)**, bukan `NaN` — sehingga tidak
terdeteksi oleh `.isnull()`. Ini dicek langsung di bawah.

In [8]:
# Coba ubah TotalCharges ke numerik; baris yang gagal dikonversi akan jadi NaN
total_charges_numeric = pd.to_numeric(df["TotalCharges"], errors="coerce")
hidden_missing_mask = total_charges_numeric.isna()

print(f"Jumlah baris dengan TotalCharges tidak valid: {hidden_missing_mask.sum()}")
df.loc[hidden_missing_mask, ["customerID", "tenure", "MonthlyCharges", "TotalCharges"]]

Jumlah baris dengan TotalCharges tidak valid: 11


,customerID,tenure,MonthlyCharges,TotalCharges
488,4472-LVYGI,0,52.55,
753,3115-CZMZD,0,20.25,
936,5709-LVOEQ,0,80.85,
1082,4367-NUYAO,0,25.75,
1340,1371-DWPAZ,0,56.05,
3331,7644-OMVMY,0,19.85,
3826,3213-VVOLG,0,25.35,
4380,2520-SGTTA,0,20.00,
5218,2923-ARZLG,0,19.70,
6670,4075-WKNIU,0,73.35,


**Temuan penting:** seluruh baris dengan `TotalCharges` tidak valid ternyata
punya `tenure = 0`, yaitu pelanggan yang baru saja mendaftar sehingga belum
punya total tagihan. Ini masuk akal secara bisnis (bukan data error acak),
dan akan ditangani secara eksplisit di **Tahap 2** (opsi: isi dengan 0, atau
`MonthlyCharges * tenure`).

## 8. Cek Duplikasi `customerID`

In [9]:
n_rows = len(df)
n_unique_ids = df["customerID"].nunique()
n_duplicate_ids = df["customerID"].duplicated().sum()

print(f"Jumlah baris          : {n_rows}")
print(f"Jumlah customerID unik: {n_unique_ids}")
print(f"Jumlah ID duplikat    : {n_duplicate_ids}")

Jumlah baris          : 7043
Jumlah customerID unik: 7043
Jumlah ID duplikat    : 0


Jika `n_duplicate_ids == 0`, berarti setiap baris memang mewakili satu pelanggan unik dan tidak perlu deduplikasi di Tahap 2.

## 9. Distribusi Target (`Churn`)

In [10]:
churn_dist = pd.DataFrame({
    "count": df["Churn"].value_counts(),
    "percentage": (df["Churn"].value_counts(normalize=True) * 100).round(2),
})
churn_dist

,count,percentage
Churn,,
No,5174,73.46
Yes,1869,26.54


Sesuai perkiraan di brief, distribusi target **tidak seimbang** (sekitar
73% `No` vs 27% `Yes`). Konsekuensinya sudah jelas untuk tahap-tahap
berikutnya:

- **Tahap 4 (Feature Engineering):** perlu strategi menangani imbalance
  (`class_weight` atau SMOTE).
- **Tahap 6 (Evaluasi):** accuracy saja tidak cukup — fokus ke recall,
  precision, F1-score, dan ROC-AUC pada kelas `Yes` (churn).

## 10. Ringkasan Tahap 1

**Yang sudah dipastikan:**

- Dataset punya 7.043 baris × 21 kolom, 1 baris = 1 pelanggan.
- Target: kolom `Churn` (Yes/No), kelas positif = `Yes`, masalah klasifikasi
  biner dengan distribusi imbalance (~27% churn).
- Tidak ada `NaN` eksplisit, tapi `TotalCharges` punya 11 baris "kosong
  tersembunyi" (string spasi) yang semuanya berasal dari pelanggan dengan
  `tenure = 0`.
- `TotalCharges` perlu dikonversi dari teks ke numerik.
- `SeniorCitizen` disimpan sebagai 0/1, sementara kolom biner lain pakai
  Yes/No — perlu diseragamkan.
- Beberapa kolom layanan punya kategori ketiga (`"No internet service"` /
  `"No phone service"`) selain Yes/No.
- `customerID` dicek keunikannya (lihat hasil bagian 8) — dibuang saat
  modeling karena bukan fitur prediktif.

**Lanjut ke Tahap 2 — Data Cleaning:**

1. Konversi `TotalCharges` ke numerik, isi/atasi 11 baris kosong (tenure=0).
2. Seragamkan `SeniorCitizen` menjadi format Yes/No (atau sebaliknya).
3. Pastikan tidak ada duplikasi `customerID` (sudah dicek di atas, tinggal
   ditindaklanjuti kalau ternyata >0).